# asyncio – Practice Exercises

Hands-on exercises for Python's `asyncio`, following `g_asyncio.ipynb`.

Run code in a Jupyter cell or an `.py` file with `asyncio.run(main())`.

### Contents

- [Exercise 1 – Event Loop and Coroutines](#exercise-1--event-loop-and-coroutines)
- [Exercise 2 – gather and sleep](#exercise-2--gather-and-sleep)
- [Exercise 3 – aiohttp Client](#exercise-3--aiohttp-client)
- [Exercise 4 – Parallel REST Calls](#exercise-4--parallel-rest-calls)
- [Exercise 5 – Timeouts and Cancellation](#exercise-5--timeouts-and-cancellation)
- [Exercise 6 – Combining with CPU-Bound Work](#exercise-6--combining-with-cpu-bound-work)
- [Exercise 7 – Concurrency Limits and Semaphore](#exercise-7--concurrency-limits-and-semaphore)
- [Exercise 8 – asyncio Best Practices](#exercise-8--asyncio-best-practices)

## Exercise 1 – Event Loop and Coroutines

**Goal**: Get comfortable defining and running coroutines.

**Requirements**:

- Define an `async def greet(name)` coroutine that prints a greeting.
- Run it with `asyncio.run()`.
- Create two coroutines and run them sequentially with `await`.

**Hint**: A coroutine does nothing until awaited or scheduled. `asyncio.run()` is the entry point.

In [3]:
import asyncio

async def greet(name: str):
    print(f"Hello {name}")

await greet("Coco")

Hello Coco


## Exercise 2 – gather and sleep

**Goal**: Run multiple coroutines concurrently and measure the speedup.

**Requirements**:

- Write `async def fetch_price(symbol, delay)` that sleeps `delay` seconds and returns a fake price.
- Use `asyncio.gather()` to fetch 5 symbols concurrently.
- Compare total elapsed time with sequential `await`.

**Hint**: `asyncio.gather(*coros)` schedules all coroutines on the event loop before any result is awaited.

In [22]:
from typing import List, Any
from pandas import Timestamp
import asyncio, aiohttp

class Binance:

    BASE_URL = "https://api.binance.com/api/v3"

    @classmethod
    def log(cls, *args):
        t = Timestamp.utcnow()
        ts = t.strftime("%H:%M:%S.%f")
        print(f"[{ts[: -3]}]", *args)
        return t

    @classmethod
    async def _request(cls, session: aiohttp.ClientSession, args: dict):

        async with session.get(**args) as response:
            try: response.raise_for_status()
            except Exception as EXC:
                cls.log(f"Response error: \"{EXC!r}\"")
                return None
            try: data = await response.json()
            except Exception as EXC:
                cls.log(f"Resp data error: \"{EXC!r}\"")
                return None
            return data

    @classmethod
    async def price(cls, symbols: List[str], timeout: float = 2):

        args = dict[str, Any](
            url = f"{cls.BASE_URL}/ticker/price",
            timeout = aiohttp.ClientTimeout(timeout))
        
        async with aiohttp.ClientSession() as session:
            tasks = dict.fromkeys(symbols)
            for symbol in symbols:
                args["params"] = {"symbol": symbol}
                future = cls._request(session, args.copy())
                tasks[symbol] = asyncio.create_task(future)
            results = dict(zip(
                symbols, await asyncio.gather(*tasks.values())))
            return {K: float(V["price"]) for K, V in results.items()}

t1 = Binance.log("requesting...")
ps = await Binance.price(["BTCUSDT", "ETHUSDT"], 4)
ls = str.join("\n", [f" - {K}: {V}" for K, V in ps.items()])
t2 = Binance.log(f"requested...\n" + ls)
delay = (t2 - t1).total_seconds()
print(f"Delay: {delay:.2f}s")

[21:35:44.949] requesting...
[21:35:45.378] requested...
 - BTCUSDT: 66119.14
 - ETHUSDT: 1988.0
Delay: 0.43s


## Exercise 3 – aiohttp Client

**Goal**: Make a real async HTTP request using `aiohttp`.

**Requirements**:

- Import `aiohttp` and create an `async with aiohttp.ClientSession() as session:` block.
- Fetch `https://httpbin.org/get` and print the JSON response.
- Handle `aiohttp.ClientError` gracefully.

**Hint**: Always use `async with` for both the session and the response to ensure proper cleanup.

In [29]:
from typing import Dict, Any
from pandas import Timestamp
import asyncio, aiohttp

class HTTPBin:

    BASE_URL = "https://httpbin.org"

    @classmethod
    def log(cls, *args):
        t = Timestamp.utcnow()
        ts = t.strftime("%H:%M:%S.%f")
        print(f"[{ts[: -3]}]", *args)
        return t

    @classmethod
    async def _request(cls, session: aiohttp.ClientSession, args: dict):

        async with session.get(**args) as response:
            try: response.raise_for_status()
            except Exception as EXC:
                cls.log(f"Response error: \"{EXC!r}\"")
                return None
            try: data = await response.json()
            except Exception as EXC:
                cls.log(f"Resp data error: \"{EXC!r}\"")
                return None
            print(data)
            return data

    @classmethod
    async def get(cls, stuff: Dict[str, Any], timeout: float = 2):

        args = dict[str, Any](url = f"{cls.BASE_URL}/get",
                timeout = aiohttp.ClientTimeout(timeout))
        
        async with aiohttp.ClientSession() as session:
            tasks = dict.fromkeys(stuff)
            for key, params in stuff.items():
                args["params"] = params
                future = cls._request(session, args.copy())
                tasks[key] = asyncio.create_task(future)
            return dict(zip(
                stuff.keys(), await asyncio.gather(*tasks.values())))

    @classmethod
    async def delay(cls, duration: float, timeout: float = 2):

        args = dict[str, Any](
            url = f"{cls.BASE_URL}/delay/{duration}",
            timeout = aiohttp.ClientTimeout(timeout))
        
        async with aiohttp.ClientSession() as session:
            return await cls._request(session, args)

t1 = Binance.log("requesting...")
stuff = {"req_1": {"A": 1, "B": 2}, "req_2": {"C": 3, "D": 4}}
ps = await HTTPBin.get(stuff)
ls = str.join("\n", [f" - {K}: {V}" for K, V in ps.items()])
t2 = Binance.log(f"requested...\n" + ls)
delay = (t2 - t1).total_seconds()
print(f"Delay: {delay:.2f}s")

[21:48:48.331] requesting...
{'args': {'C': '3', 'D': '4'}, 'headers': {'Accept': '*/*', 'Accept-Encoding': 'gzip, deflate', 'Host': 'httpbin.org', 'User-Agent': 'Python/3.13 aiohttp/3.12.13', 'X-Amzn-Trace-Id': 'Root=1-69c6fb41-35edefd32eb74793438e09a4'}, 'origin': '201.216.219.254', 'url': 'https://httpbin.org/get?C=3&D=4'}
{'args': {'A': '1', 'B': '2'}, 'headers': {'Accept': '*/*', 'Accept-Encoding': 'gzip, deflate', 'Host': 'httpbin.org', 'User-Agent': 'Python/3.13 aiohttp/3.12.13', 'X-Amzn-Trace-Id': 'Root=1-69c6fb41-32746b96253b057c198f97c4'}, 'origin': '201.216.219.254', 'url': 'https://httpbin.org/get?A=1&B=2'}
[21:48:48.957] requested...
 - req_1: {'args': {'A': '1', 'B': '2'}, 'headers': {'Accept': '*/*', 'Accept-Encoding': 'gzip, deflate', 'Host': 'httpbin.org', 'User-Agent': 'Python/3.13 aiohttp/3.12.13', 'X-Amzn-Trace-Id': 'Root=1-69c6fb41-32746b96253b057c198f97c4'}, 'origin': '201.216.219.254', 'url': 'https://httpbin.org/get?A=1&B=2'}
 - req_2: {'args': {'C': '3', 'D': '

## Exercise 4 – Parallel REST Calls

**Goal**: Fetch multiple endpoints concurrently and aggregate results.

**Requirements**:

- Build a list of URLs (e.g. `https://httpbin.org/delay/1` twice and `https://httpbin.org/get` once).
- Fetch them all with `asyncio.gather()` inside a single `ClientSession`.
- Print each response status and elapsed time.

**Hint**: Reuse one `ClientSession` across all requests for efficiency.

In [32]:
async def parallel():
    stuff = {"req_1": {"A": 1, "B": 2}, "req_2": {"C": 3, "D": 4}}
    return await asyncio.gather(HTTPBin.get(stuff), HTTPBin.delay(1),
        return_exceptions = True)

await parallel()

{'args': {'A': '1', 'B': '2'}, 'headers': {'Accept': '*/*', 'Accept-Encoding': 'gzip, deflate', 'Host': 'httpbin.org', 'User-Agent': 'Python/3.13 aiohttp/3.12.13', 'X-Amzn-Trace-Id': 'Root=1-69c6fbce-7de1cf1d63e459eb7ee44834'}, 'origin': '201.216.219.254', 'url': 'https://httpbin.org/get?A=1&B=2'}
{'args': {'C': '3', 'D': '4'}, 'headers': {'Accept': '*/*', 'Accept-Encoding': 'gzip, deflate', 'Host': 'httpbin.org', 'User-Agent': 'Python/3.13 aiohttp/3.12.13', 'X-Amzn-Trace-Id': 'Root=1-69c6fbce-6fa3fffe70f839834cf34d32'}, 'origin': '201.216.219.254', 'url': 'https://httpbin.org/get?C=3&D=4'}
{'args': {}, 'data': '', 'files': {}, 'form': {}, 'headers': {'Accept': '*/*', 'Accept-Encoding': 'gzip, deflate', 'Host': 'httpbin.org', 'User-Agent': 'Python/3.13 aiohttp/3.12.13', 'X-Amzn-Trace-Id': 'Root=1-69c6fbce-1f7c588b28f6b67c4ed7355a'}, 'origin': '201.216.219.254', 'url': 'https://httpbin.org/delay/1'}


[{'req_1': {'args': {'A': '1', 'B': '2'},
   'headers': {'Accept': '*/*',
    'Accept-Encoding': 'gzip, deflate',
    'Host': 'httpbin.org',
    'User-Agent': 'Python/3.13 aiohttp/3.12.13',
    'X-Amzn-Trace-Id': 'Root=1-69c6fbce-7de1cf1d63e459eb7ee44834'},
   'origin': '201.216.219.254',
   'url': 'https://httpbin.org/get?A=1&B=2'},
  'req_2': {'args': {'C': '3', 'D': '4'},
   'headers': {'Accept': '*/*',
    'Accept-Encoding': 'gzip, deflate',
    'Host': 'httpbin.org',
    'User-Agent': 'Python/3.13 aiohttp/3.12.13',
    'X-Amzn-Trace-Id': 'Root=1-69c6fbce-6fa3fffe70f839834cf34d32'},
   'origin': '201.216.219.254',
   'url': 'https://httpbin.org/get?C=3&D=4'}},
 {'args': {},
  'data': '',
  'files': {},
  'form': {},
  'headers': {'Accept': '*/*',
   'Accept-Encoding': 'gzip, deflate',
   'Host': 'httpbin.org',
   'User-Agent': 'Python/3.13 aiohttp/3.12.13',
   'X-Amzn-Trace-Id': 'Root=1-69c6fbce-1f7c588b28f6b67c4ed7355a'},
  'origin': '201.216.219.254',
  'url': 'https://httpbin.or

## Exercise 5 – Timeouts and Cancellation

**Goal**: Enforce per-request time budgets and handle cancellation.

**Requirements**:

- Wrap a slow coroutine with `asyncio.wait_for(coro, timeout=...)` and catch `asyncio.TimeoutError`.
- Create a `Task` with `asyncio.create_task()` and call `.cancel()` on it; catch `asyncio.CancelledError`.
- Show that cancelled tasks do not produce results.

**Hint**: `asyncio.wait_for` wraps any awaitable; `create_task` is needed to cancel mid-flight.

In [44]:
class Binance:

    BASE_URL = "https://api.binance.com/api/v3"

    @classmethod
    def log(cls, *args):
        t = Timestamp.utcnow()
        ts = t.strftime("%H:%M:%S.%f")
        print(f"[{ts[: -3]}]", *args)
        return t

    @classmethod
    async def _request(cls, session: aiohttp.ClientSession, args: dict):

        async with session.get(**args) as response:
            try: response.raise_for_status()
            except Exception as EXC:
                cls.log(f"Response error: \"{EXC!r}\"")
                return None
            try: data = await response.json()
            except Exception as EXC:
                cls.log(f"Resp data error: \"{EXC!r}\"")
                return None
            return data

    @classmethod
    async def price(cls, symbols: List[str], timeout: float = 2):

        args = dict[str, Any](
            url = f"{cls.BASE_URL}/ticker/price")
        
        async with aiohttp.ClientSession() as session:
            tasks = dict.fromkeys(symbols)
            for symbol in symbols:
                args["params"] = {"symbol": symbol}
                future = cls._request(session, args.copy())
                task = asyncio.create_task(future)
                tasks[symbol] = asyncio.wait_for(task, timeout = timeout)
            results = dict(zip(
                symbols, await asyncio.gather(*tasks.values(),
                    return_exceptions = True)))
            for K, V in results.items():
                if isinstance(V, Exception):
                    results[K] = repr(V)
                else: results[K] = float(V["price"])
            return results

t1 = Binance.log("requesting...")
ps = await Binance.price(["BTCUSDT", "ETHUSDT"], 0.42)
ls = str.join("\n", [f" - {K}: {V}" for K, V in ps.items()])
t2 = Binance.log(f"requested...\n" + ls)
delay = (t2 - t1).total_seconds()
print(f"Delay: {delay:.2f}s")

[22:04:01.888] requesting...
[22:04:02.327] requested...
 - BTCUSDT: TimeoutError()
 - ETHUSDT: TimeoutError()
Delay: 0.44s


## Exercise 6 – Combining with CPU-Bound Work

**Goal**: Offload blocking computation without stalling the event loop.

**Requirements**:

- Write a CPU-bound function `slow_compute(n)` (e.g. sum of squares up to n=10_000_000).
- Call it with `loop.run_in_executor(None, slow_compute, n)` inside an async context.
- Verify other coroutines keep running while the compute is offloaded.

**Hint**: Pass `None` as executor to use the default `ThreadPoolExecutor`. Use `ProcessPoolExecutor` for GIL-bound work.

## Exercise 7 – Concurrency Limits and Semaphore

**Goal**: Prevent overwhelming a server by capping simultaneous requests.

**Requirements**:

- Create `asyncio.Semaphore(3)` to allow at most 3 concurrent fake fetches.
- Launch 10 tasks; each acquires the semaphore before running and releases it after.
- Print when each task starts and ends to observe the batching effect.

**Hint**: `async with semaphore:` automatically acquires and releases.

In [59]:
SYMBOLS = sorted(["BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "XRPUSDT", "ADAUSDT",
    "DOGEUSDT", "AVAXUSDT", "DOTUSDT", "LINKUSDT", "MATICUSDT", "UNIUSDT", "LTCUSDT",
    "ATOMUSDT", "ICPUSDT", "APTUSDT", "ETCUSDT", "FILUSDT", "NEARUSDT", "OPUSDT",
    "STXUSDT", "RNDRUSDT", "INJUSDT", "SHIBUSDT", "PEPEUSDT", "ARBUSDT", "IMXUSDT",
    "LDOUSDT", "TIAUSDT", "SEIUSDT", "SUIUSDT", "FLOWUSDT", "AXSUSDT", "CHZUSDT",
    "SANDUSDT", "MANAUSDT", "AAVEUSDT", "CRVUSDT", "MKRUSDT", "COMPUSDT", "SNXUSDT",
    "DYDXUSDT", "FETUSDT", "AGIXUSDT", "OCEANUSDT", "GALAUSDT", "ENJUSDT", "YFIUSDT",
    "SUSHIUSDT", "CAKEUSDT", ])

SYMBOLS = [s.replace("USDT", "") for s in SYMBOLS]
str.join(" ", SYMBOLS)

'AAVE ADA AGIX APT ARB ATOM AVAX AXS BNB BTC CAKE CHZ COMP CRV DOGE DOT DYDX ENJ ETC ETH FET FIL FLOW GALA ICP IMX INJ LDO LINK LTC MANA MATIC MKR NEAR OCEAN OP PEPE RNDR SAND SEI SHIB SNX SOL STX SUI SUSHI TIA UNI XRP YFI'

In [61]:
from typing import List, Any
from pandas import Timestamp
import asyncio, aiohttp

class Binance:

    BASE_URL = "https://api.binance.com/api/v3"

    @classmethod
    def log(cls, *args):
        t = Timestamp.utcnow()
        ts = t.strftime("%H:%M:%S.%f")
        print(f"[{ts[: -3]}]", *args)
        return t

    @classmethod
    async def _request(cls, session: aiohttp.ClientSession,
                semaphore: asyncio.Semaphore, args: dict):

        async with semaphore:
            async with session.get(**args) as response:
                try: response.raise_for_status()
                except Exception as EXC:
                    cls.log(f"Response error: \"{EXC!r}\"")
                    return None
                try: data = await response.json()
                except Exception as EXC:
                    cls.log(f"Resp data error: \"{EXC!r}\"")
                    return None
                return data

    @classmethod
    async def price(cls, symbols: List[str], timeout: float = 2, max_threads: int = 4):

        args = dict[str, Any](
            url = f"{cls.BASE_URL}/ticker/price",
            timeout = aiohttp.ClientTimeout(timeout))
        
        semaphore = asyncio.Semaphore(max_threads)
        async with aiohttp.ClientSession() as session:
            tasks = dict.fromkeys(symbols)
            for symbol in symbols:
                args["params"] = {"symbol": symbol}
                future = cls._request(session, semaphore, args.copy())
                tasks[symbol] = asyncio.create_task(future)
            results = dict(zip(
                symbols, await asyncio.gather(*tasks.values())))
            return {K: float(V["price"]) for K, V in results.items()}


SYMBOLS = """AAVE ADA AGIX APT ARB ATOM AVAX AXS BNB BTC CAKE CHZ COMP CRV DOGE DOT
    DYDX ENJ ETC ETH FET FIL FLOW GALA ICP IMX INJ LDO LINK LTC MANA MATIC MKR NEAR
    OCEAN OP PEPE RNDR SAND SEI SHIB SNX SOL STX SUI SUSHI TIA UNI XRP YFI"""

symbols = [(s + "USDT") for s in SYMBOLS.replace("\n", " ").split(" ")]

t1 = Binance.log("requesting...")
ps = await Binance.price(symbols, timeout = 10, max_threads = 4)
ls = str.join("\n", [f" - {K}: {V}" for K, V in ps.items()])
t2 = Binance.log(f"requested...\n" + ls)
delay = (t2 - t1).total_seconds()
print(f"Delay: {delay:.2f}s")

[22:20:12.591] requesting...
[22:20:16.081] Response error: "ClientResponseError(RequestInfo(url=URL('https://api.binance.com/api/v3/ticker/price?symbol=USDT'), method='GET', headers=<CIMultiDictProxy('Host': 'api.binance.com', 'Accept': '*/*', 'Accept-Encoding': 'gzip, deflate', 'User-Agent': 'Python/3.13 aiohttp/3.12.13')>, real_url=URL('https://api.binance.com/api/v3/ticker/price?symbol=USDT')), (), status=400, message='Bad Request', headers=<CIMultiDictProxy('Content-Type': 'application/json;charset=UTF-8', 'Content-Length': '38', 'Connection': 'keep-alive', 'Date': 'Fri, 27 Mar 2026 22:20:16 GMT', 'Expires': '0', 'Server': 'nginx', 'x-mbx-uuid': 'e49bd0cd-fa1b-43e3-9c5e-4237266a2871', 'x-mbx-used-weight': '34', 'x-mbx-used-weight-1m': '34', 'Strict-Transport-Security': 'max-age=31536000; includeSubdomains', 'X-Frame-Options': 'SAMEORIGIN', 'X-Xss-Protection': '1; mode=block', 'X-Content-Type-Options': 'nosniff', 'Content-Security-Policy': "default-src 'self'", 'X-Content-Security-

## Exercise 8 – asyncio Best Practices

**Goal**: Demonstrate common pitfalls and how to avoid them.

**Requirements**:

- Show what happens when you call a sync-blocking `time.sleep()` inside a coroutine instead of `await asyncio.sleep()`.
- Demonstrate that creating a coroutine object without awaiting it silently does nothing (and how to detect this).
- Write a clean `main()` that uses `asyncio.run()` exactly once at the top level.

**Hint**: Use `asyncio.get_event_loop().set_debug(True)` to get warnings about unawaited coroutines.

In [65]:
import asyncio
import time

async def blocking_sleep():
    print("About to block with time.sleep(1)...")
    time.sleep(1)  # This blocks the event loop!
    print("Finished blocking.")

async def nonblocking_sleep():
    print("About to await asyncio.sleep(1)...")
    await asyncio.sleep(1)
    print("Finished non-blocking sleep.")

async def silent_coroutine():
    print("I am never awaited!")

async def main():
    # 1. Show time.sleep() blocks event loop
    print("== Blocking call inside coroutine ==")
    t1 = time.perf_counter()
    await blocking_sleep()
    t2 = time.perf_counter()
    print(f"Elapsed with time.sleep: {t2 - t1:.2f}s (should be ~1s and blocks whole event loop)\n")

    # 2. Show proper way of sleeping in coroutines
    print("== Non-blocking asyncio.sleep ==")
    t3 = time.perf_counter()
    await nonblocking_sleep()
    t4 = time.perf_counter()
    print(f"Elapsed with asyncio.sleep: {t4 - t3:.2f}s\n")

    # 3. Silent coroutine demonstration
    print("== Unawaited coroutine is not scheduled ==")
    c = silent_coroutine()
    print("Created a coroutine object, but did not await() or schedule it.")
    
# To see warnings for this
loop = asyncio.get_event_loop()
loop.set_debug(True)
# Will show warnings for un-awaited coroutines at program exit
try: await main()
except Exception as EXC:
    print(repr(EXC))


== Blocking call inside coroutine ==
About to block with time.sleep(1)...
Finished blocking.
Elapsed with time.sleep: 1.00s (should be ~1s and blocks whole event loop)

== Non-blocking asyncio.sleep ==
About to await asyncio.sleep(1)...
Finished non-blocking sleep.
Elapsed with asyncio.sleep: 1.02s

== Unawaited coroutine is not scheduled ==
Created a coroutine object, but did not await() or schedule it.


C:\Users\GSL\AppData\Local\Temp\ipykernel_48596\1466267743.py:41: RuntimeWarning: coroutine 'silent_coroutine' was never awaited
Coroutine created at (most recent call last)
  File "c:\DEV\Python\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.start()
  File "c:\DEV\Python\Lib\site-packages\tornado\platform\asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "c:\DEV\Python\Lib\asyncio\base_events.py", line 683, in run_forever
    self._run_once()
  File "c:\DEV\Python\Lib\asyncio\base_events.py", line 2034, in _run_once
    handle._run()
  File "c:\DEV\Python\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
  File "c:\DEV\Python\Lib\site-packages\IPython\core\interactiveshell.py", line 3367, in run_cell_async
    has_raised = await self.run_ast_nodes(code_ast.body, cell_name,
  File "c:\DEV\Python\Lib\site-packages\IPython\core\interactiveshell.py", line 3612, in run_ast_nodes
    if await 